# Benchmark Dataset Tutorial

Load the prepared AgentDojo and InjecAgent benchmark JSONL files.

In [ ]:
from collections import Counter
from pathlib import Path
import sys

from datasets import load_dataset


def find_repo_root(start: Path) -> Path:
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "authority_data" / "__init__.py").exists():
            return path
    raise RuntimeError("Could not find the authority-data repository root.")


ROOT = find_repo_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

benchmark_root = ROOT / "authority_data" / "data" / "benchmarks"


## AgentDojo

In [ ]:
agentdojo_dir = benchmark_root / "agentdojo"
agentdojo_configs = ["permission", "prohibition", "permission_and_prohibition"]

agentdojo_datasets = {
    config_name: load_dataset(
        "json",
        data_files={
            "train": str(agentdojo_dir / config_name / "train.jsonl"),
            "test": str(agentdojo_dir / config_name / "test.jsonl"),
        },
    )
    for config_name in agentdojo_configs
}
agentdojo_datasets


In [ ]:
for config_name, dataset_dict in agentdojo_datasets.items():
    print(f"[agentdojo/{config_name}]")
    for split in ["train", "test"]:
        rows = dataset_dict[split]
        print(
            split,
            "rows=", rows.num_rows,
            "labels=", dict(Counter(rows["label"])),
            "decision_kind=", dict(Counter(rows["decision_kind"])),
        )
    overlap = set(dataset_dict["train"]["source_sample_id"]) & set(dataset_dict["test"]["source_sample_id"])
    print("source_sample_id_overlap=", len(overlap))
    print()


In [ ]:
agentdojo_datasets["permission"]["train"][0]

In [ ]:
print(agentdojo_datasets["permission"]["train"][0]["prompt_v1"])

## InjecAgent

In [ ]:
injecagent_dir = benchmark_root / "injecagent"
injecagent_configs = [
    "base_direct_harm",
    "base_data_stealing",
    "enhanced_direct_harm",
    "enhanced_data_stealing",
]

injecagent_datasets = {
    config_name: load_dataset(
        "json",
        data_files={
            "train": str(injecagent_dir / config_name / "train.jsonl"),
            "test": str(injecagent_dir / config_name / "test.jsonl"),
        },
    )
    for config_name in injecagent_configs
}
injecagent_datasets


In [ ]:
for config_name, dataset_dict in injecagent_datasets.items():
    print(f"[injecagent/{config_name}]")
    for split in ["train", "test"]:
        rows = dataset_dict[split]
        print(
            split,
            "rows=", rows.num_rows,
            "labels=", dict(Counter(rows["label"])),
            "action_type=", dict(Counter(rows["action_type"])),
        )
    print()


In [ ]:
first_config = next(iter(injecagent_datasets))
print(injecagent_datasets[first_config]["train"][0]["prompt_v1"])
